# Policy gradients: optimize the policy directly by gradient ascent on return

_Reinforcement Learning_

**Don't learn a value table and act greedily — parameterize the policy and climb the expected-return hill, pushing up the log-probability of actions that paid off.**

Value-based RL takes a detour: learn how good each action is ($Q$), then act greedily.
       Policy gradient skips the detour &mdash; it writes the policy as a differentiable
       function $\pi_\theta(a\mid s)$ with parameters $\theta$ (e.g. a neural network), and turns
       "find a good policy" into a plain optimization problem: maximize expected return
       $J(\theta)=\mathbb{E}_{\pi_\theta}[G]$ by gradient ascent,
       $\theta \leftarrow \theta + \alpha\,\nabla_\theta J(\theta)$.

       The obstacle is that $J$ is an average over trajectories the policy itself
       generates &mdash; change $\theta$ and you change which data you see, so it is not obvious
       you can differentiate through it by sampling. The policy-gradient theorem resolves
       this. It says the gradient is itself an expectation you can estimate from samples:
       $\nabla_\theta J(\theta)=\mathbb{E}_{\pi_\theta}\!\big[\nabla_\theta\log\pi_\theta(a\mid s)\,Q^{\pi}(s,a)\big]$.

---

This notebook is a practice scaffold for the **Policy gradients: optimize the policy directly by gradient ascent on return** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch

### Step 1 — Build the policy network and the environment

Policy gradients parameterize the policy directly as a differentiable function `pi_theta(a|s)` and climb the expected-return hill. Here that policy is a small two-layer network whose softmax output gives action probabilities for CartPole (4-dim state, 2 actions). We pair it with an Adam optimizer and a discount factor `gamma = 0.99`.

In [ ]:
# Colab:  !pip install gymnasium torch
import gymnasium as gym
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np

torch.manual_seed(0); np.random.seed(0)
env = gym.make("CartPole-v1")
obs_dim = env.observation_space.shape[0]   # 4: cart pos/vel, pole angle/vel
n_act   = env.action_space.n               # 2: push left / push right
gamma   = 0.99

# ---- Policy network: state -> action probabilities (softmax) ----
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, 128)
        self.fc2 = nn.Linear(128, n_act)
    def forward(self, s):
        h = F.relu(self.fc1(s))
        return F.softmax(self.fc2(h), dim=-1)   # pi_theta(a | s)

policy = Policy()
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

### Step 2 — Collect an episode and compute returns-to-go

The policy-gradient estimate is an expectation over trajectories the policy itself generates, so we need to **sample** episodes. `run_episode` samples an action from the categorical distribution at each step and records its `log pi(a|s)` together with the reward. `discounted_returns` then turns the reward sequence into returns-to-go `G_t = sum_{k>=t} gamma^{k-t} r_k`, computed backwards in one pass.

In [ ]:
def run_episode():
    """Collect one episode: returns lists of log pi(a|s) and rewards."""
    s, _ = env.reset()
    log_probs, rewards = [], []
    done = False
    while not done:
        s_t = torch.as_tensor(s, dtype=torch.float32)
        probs = policy(s_t)                     # action probabilities
        dist  = torch.distributions.Categorical(probs)
        a     = dist.sample()                   # SAMPLE the action
        log_probs.append(dist.log_prob(a))      # log pi_theta(a | s)
        s, r, term, trunc, _ = env.step(a.item())
        rewards.append(r)
        done = term or trunc
    return log_probs, rewards

def discounted_returns(rewards):
    """Returns-to-go G_t = sum_{k>=t} gamma^{k-t} r_k (computed backward)."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.insert(0, G)
    return torch.tensor(out, dtype=torch.float32)

### Step 3 — Train with REINFORCE and a baseline

This is the learning loop. Each episode we form the REINFORCE loss `-sum_t log pi(a_t|s_t) * G_t`; minimizing it is the same as ascending the expected return `J(theta)`. Standardizing the returns (subtract mean, divide by std) is the simplest possible **baseline** — it does not bias the gradient but sharply cuts its variance. Watch the printed return climb toward CartPole's ~475 "solved" threshold; try `USE_BASELINE = False` to see learning get noisier.

In [ ]:
USE_BASELINE = True   # standardize returns -> simplest baseline (cuts variance)

for ep in range(600):
    log_probs, rewards = run_episode()
    G = discounted_returns(rewards)
    if USE_BASELINE:                            # subtract baseline (mean), scale
        G = (G - G.mean()) / (G.std() + 1e-8)
    # REINFORCE loss:  -sum_t log pi_theta(a_t | s_t) * G_t
    # (minimizing -log_prob * G  ==  ascending J(theta))
    loss = -(torch.stack(log_probs) * G).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    if ep % 50 == 0:
        print(f"episode {ep:4d}   return = {sum(rewards):6.1f}")

# CartPole is 'solved' at an average return of ~475 over 100 episodes.
# Try USE_BASELINE = False to watch learning get noticeably noisier.

## Visualize the data & results

_Does a REINFORCE-style policy gradient actually raise the return over training, and does subtracting a baseline reduce the variance of the learning curve? We implement a tabular softmax policy gradient (numpy only) on a tiny 3-action bandit and plot the average reward per episode, WITH versus WITHOUT a baseline._

### Step 1 — Set up a tiny bandit and a softmax policy

To isolate the effect of a baseline we drop to the simplest case: a one-state, 3-action bandit whose rewards have a **large positive offset** (means 8, 9, 10). That offset is exactly the regime where a baseline matters most — without it, every sampled action gets pushed up. The policy is a softmax over learnable action preferences (logits).

In [ ]:
import numpy as np

# Tiny 3-action bandit (a one-state MDP). Rewards are stochastic AND have a
# large positive offset -- the regime where a baseline matters most: without
# it, every sampled action is pushed UP, slowing learning.
means = np.array([8.0, 9.0, 10.0])   # arm 2 is best (expected reward 10.0)
A, alpha, EPISODES, RUNS = 3, 0.02, 1000, 60

def softmax(theta):
    z = theta - theta.max()
    e = np.exp(z)
    return e / e.sum()

### Step 2 — Train with the softmax policy gradient, optionally with a baseline

Each episode samples an action from the softmax, observes a noisy reward, and forms the advantage `r - b`, where `b` is a running average reward (the baseline) — or `0` when disabled. The softmax score `d/dtheta log pi(a) = 1{i==a} - p[i]` gives the gradient direction, and we take an ascent step scaled by the advantage. The running baseline is updated toward the latest reward.

In [ ]:
def train(use_baseline, seed):
    rng = np.random.default_rng(seed)
    theta = np.zeros(A)              # action preferences (logits)
    baseline = 0.0                   # running average reward (the baseline b)
    rewards = np.empty(EPISODES)
    for t in range(EPISODES):
        p = softmax(theta)
        a = rng.choice(A, p=p)                      # sample action ~ pi
        r = means[a] + rng.normal(0, 1.0)          # noisy reward
        adv = r - (baseline if use_baseline else 0.0)   # advantage = r - b(s)
        # score of softmax: d/dtheta log pi(a) = 1{i==a} - p[i]
        grad = -p.copy(); grad[a] += 1.0
        theta += alpha * adv * grad                 # gradient ASCENT on J
        baseline += 0.05 * (r - baseline)           # update running baseline
        rewards[t] = r
    return rewards

### Step 3 — Average over many runs and compare the learning curves

A single run is too noisy to judge, so we average over 60 seeds and then into blocks of 50 episodes for 20 plotted points. Comparing the curves **with** versus **without** the baseline shows the payoff: subtracting the baseline removes the shared positive offset, so only better-than-average actions get reinforced, and the policy locks onto the best arm faster and more steadily.

In [ ]:
# average over RUNS seeds, then average into blocks of 50 episodes
def curve(use_baseline):
    allr = np.mean([train(use_baseline, s) for s in range(RUNS)], axis=0)
    return allr.reshape(-1, 50).mean(axis=1)        # 20 plotted points

print("with baseline:", np.round(curve(True), 3))
print("no baseline:  ", np.round(curve(False), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why is the policy gradient model-free &mdash; i.e. why does it not require knowing the transition dynamics $P(s'\mid s,a)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the log-probability of a trajectory. — _$\log P_\theta(\tau)=\log\rho(s_0)+\sum_t[\log\pi_\theta(a_t\mid s_t)+\log P(s_{t+1}\mid s_t,a_t)]$._
- Differentiate with respect to $\theta$. — _The start distribution $\rho(s_0)$ and every dynamics term $P(s_{t+1}\mid s_t,a_t)$ do not depend on $\theta$, so their gradients are zero._
- Read off what survives. — _Only $\sum_t\nabla_\theta\log\pi_\theta(a_t\mid s_t)$ remains &mdash; just the policy, which we know in closed form._

**Answer:** Because in $\nabla_\theta\log P_\theta(\tau)$ the dynamics and start-state terms are constants in $\theta$ and vanish, leaving only $\nabla_\theta\log\pi_\theta$. You never differentiate (or even need) $P(s'\mid s,a)$ &mdash; you only need to sample from the world, not model it.

</details>

**Problem 2.** A REINFORCE update uses a sampled return $G_t=+10$. A teammate subtracts the state-value $V(s)=+9$ first and uses the advantage instead. Does this bias the gradient, and what changes?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the baseline result. — _For any $b(s)$ depending only on the state, $\mathbb{E}_{a\sim\pi}[\nabla_\theta\log\pi_\theta(a\mid s)\,b(s)]=0$, because $\mathbb{E}_{a\sim\pi}[\nabla_\theta\log\pi_\theta(a\mid s)]=0$._
- Conclude on bias. — _Subtracting $b(s)$ removes a term whose expectation is zero, so the gradient's mean is unchanged &mdash; unbiased._
- Conclude on variance. — _The weight shrinks from $G_t=10$ to the advantage $A=G_t-V=1$, a small number fluctuating around zero, so the estimator's variance drops sharply._

**Answer:** No bias &mdash; the expected gradient is identical. But the multiplier changes from $10$ to the advantage $1$, a much smaller, mean-zero-ish quantity, so the variance of the estimate falls. That is precisely why baselines (and actor-critic) make learning faster and steadier.

</details>

**Problem 3.** Your CartPole policy network outputs action probabilities, but training diverges: the loss explodes after a few good episodes. The returns range up to $+500$. Name two likely causes and their fixes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the reward magnitude. — _The update scales with the return; returns near $500$ make each gradient step huge, so a couple of long episodes can blow up $\theta$._
- Look at variance / step size. — _Plain REINFORCE has high variance and is sensitive to the learning rate; a large step on a noisy gradient can collapse or explode the policy._
- Apply the standard fixes. — _Standardize returns per batch (subtract mean, divide by std), and/or subtract a baseline $V(s)$; lower the learning rate and add an entropy bonus._

**Answer:** (1) Reward scaling: returns up to $500$ make the gradient enormous &mdash; standardize returns per batch. (2) High variance / large steps: subtract a baseline (use the advantage) and lower the learning rate; an entropy bonus prevents premature collapse. These are exactly the policy-gradient pitfalls, with their fixes.

</details>